# Phase lead of sign-invariant z interventions

This notebook reads intervention tables already saved by notebook 19. It does **not** run OSSE, DA, or CSE and does not change any intervention. For each applied push it starts a short, unforced Lorenz-63 forecast from the **controlled nature state immediately before the push**. It compares the saved sign of $u_z(t)$ with the future sign of $q(t+\tau)=x(t+\tau)\dot x(t+\tau)$. Under $(x,y,z)\mapsto(-x,-y,z)$, $q$ is unchanged.

This is a retrospective test using true states, **not** a rule available to the z-only controller. Each forecast omits the current push and every later push, avoiding feedback from that push into the future-phase diagnostic. Correlations across thousands of consecutive pushes are not independent observations; compare the curves across events and lobes before interpreting a peak.


In [ ]:
from pathlib import Path

from class_da_sys import da_system
from cse_phase_lead_diagnostic import summarize_phase_lead, plot_phase_lead
from project_paths import ANALYSIS_DATA_DIR, CSE_ROOT

EXPERIMENT_ROOT = CSE_ROOT / 'z_control_z_obs_sign_invariant_qiwenevents'
ANALYSIS_FILE = (
    ANALYSIS_DATA_DIR.parent / 'z_only_obs' / 'data'
    / 'x_analysis_init_z_only_obs.pkl'
)
das = da_system().load(ANALYSIS_FILE)
DT = float(das.fcst_dt)
TA = int(das.acyc_step)
MAX_LEAD_STEPS = TA  # initially test zero through one DA cycle
D_TAG = '0p05'

def intervention_table(lobe, event_index):
    return (
        EXPERIMENT_ROOT / 'results' / f'initial_nature_{lobe}'
        / ('sign_of_uz_vs_pre_intervention_state_z_obs_sign_invariant_'
           f'event_{event_index}_D{D_TAG}_qiwenevents.csv')
    )

EVENT_TABLES = {
    'positive event 0': intervention_table('positive', 0),
    'positive event 1': intervention_table('positive', 1),
    'negative event 0': intervention_table('negative', 0),
}
for label, path in EVENT_TABLES.items():
    if not path.is_file():
        raise FileNotFoundError(f'{label}: {path}')
print(f'Model step dt = {DT:g}; DA cycle = {TA} model steps')
for label, path in EVENT_TABLES.items():
    print(f'{label}: {path}')


In [ ]:
phase_lead_summary = summarize_phase_lead(
    EVENT_TABLES, dt=DT, max_lead_steps=MAX_LEAD_STEPS
)

# The lead-zero column must reproduce the previously reported same-time
# agreement. Keep counts visible so undefined phases cannot be hidden.
print('Event                 Lead  Pushes  Defined  Undefined  Agree  Agreement')
for label in EVENT_TABLES:
    rows = [r for r in phase_lead_summary
            if r['event'] == label and r['branch'] == 'all']
    for r in rows:
        print(f"{label:<21} {r['lead_steps']:>4} {r['n_pushes']:>7} "
              f"{r['n_defined']:>8} {r['n_undefined']:>10} "
              f"{r['n_agree']:>6} {r['agreement_percent']:>9.2f}%")
    best = max(rows, key=lambda r: r['agreement_percent'])
    print(f"  Descriptive maximum: lead {best['lead_steps']} "
          f"({best['lead_time']:.3f} time units), "
          f"{best['agreement_percent']:.2f}%")

print('\nBranch-specific lead-zero agreement and descriptive maximum:')
for label in EVENT_TABLES:
    for branch in ('current_safe_control', 'past_safe_control'):
        rows = [r for r in phase_lead_summary if r['event'] == label
                and r['branch'] == branch]
        if not rows:
            continue
        first = rows[0]
        best = max(rows, key=lambda r: r['agreement_percent'])
        print(f"{label:<21} {branch:<20} "
              f"lead 0: {first['n_agree']}/{first['n_defined']} "
              f"({first['agreement_percent']:.2f}%); "
              f"maximum at lead {best['lead_steps']}: "
              f"{best['agreement_percent']:.2f}%")


In [ ]:
from IPython.display import Image, display

figure_path = (
    EXPERIMENT_ROOT / 'figures'
    / f'uz_sign_vs_unforced_future_phase_D{D_TAG}_three_events.png'
)
figure_path = plot_phase_lead(phase_lead_summary, figure_path)
print('Figure:', figure_path)
display(Image(filename=str(figure_path)))


## Interpretation limits

A peak at a positive lead would be consistent with an anticipatory direction choice; it would not prove the controller uses this rule. The future phase here comes from short forecasts initialized with **true** pre-push states. The next, separate test would estimate the same invariant from available analysis-ensemble forecasts and validate any chosen lead on events not used to select it. Positive event 1 may include pushes after a true lobe crossing, so its label refers only to the **initial** nature lobe.
